In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import pandas as pd

In [15]:
driver = webdriver.Chrome()

driver.get("https://www.cardekho.com/filter/new-cars")

wait = WebDriverWait(driver, 20)

CAR_SELECTOR = ".gsc_col-sm-7.gsc_col-xs-8.gsc_col-md-8.listView.holder"
TARGET_CAR_COUNT = 293 
# set limit because this page contains 293 car details 
#Load cars (up to TARGET_CAR_COUNT)

prev_count = 0
same_count_streak = 0
max_clicks = 20 
# hard safety cap so it can NEVER loop forever


for _ in range(max_clicks):
    current_count = len(driver.find_elements(By.CSS_SELECTOR, CAR_SELECTOR))
    print(f"Cars loaded so far: {current_count}")

    if current_count >= TARGET_CAR_COUNT:
        print(f"Reached target of {TARGET_CAR_COUNT} cars. Stopping.")
        break

    if current_count == prev_count:
        same_count_streak += 1
        if same_count_streak >= 5:   
            print("Car count stopped increasing. Done loading.")
            break
    else:
        same_count_streak = 0

    prev_count = current_count

    try:
        load_more = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".showMoreButton"))
        )
        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            load_more
        )
        time.sleep(0.5)
        driver.execute_script(
            "arguments[0].click();",
            load_more
        )
        time.sleep(2)
    except TimeoutException:
        print("Load More button not clickable/found — assuming end of list.")
        break
    except (ElementClickInterceptedException, StaleElementReferenceException):
        print("Click intercepted or stale element — retrying next loop.")
        continue

        
# Find all cars (trim to target count)


cars = driver.find_elements(By.CSS_SELECTOR, CAR_SELECTOR)[:TARGET_CAR_COUNT]
print("Total cars to extract:", len(cars))

# Extract data


stored = []
for car in cars:
    try:
        name = car.find_element(By.TAG_NAME, "h3").text
    except NoSuchElementException:
        name = None
    try:
        rating = car.find_element(
            By.CSS_SELECTOR, ".startRating .ratingStarNew"
        ).text
    except NoSuchElementException:
        rating = None
    try:
        price = car.find_element(By.CSS_SELECTOR, ".price").text
    except NoSuchElementException:
        price = None
    try:
        mileage = car.find_element(
            By.CSS_SELECTOR, '.dotlist span[title="Mileage"]'
        ).text
    except NoSuchElementException:
        mileage = None
    try:
        engine = car.find_element(
            By.CSS_SELECTOR, '.dotlist span[title^="Engine Displacement"]'
        ).text
    except NoSuchElementException:
        engine = None

    car_details = {
        "car_name": name,
        "rating": rating,
        "price": price,
        "mileage": mileage,
        "engine": engine
    }
    stored.append(car_details)
print("\n Extraction done!")
driver.quit()

df = pd.DataFrame(stored)

df.to_csv("cardhekho_new_cars.csv",index=False)

print("\nsaved as cardhekho_new_cars.csv")

Cars loaded so far: 21
Cars loaded so far: 21
Cars loaded so far: 42
Cars loaded so far: 63
Cars loaded so far: 84
Cars loaded so far: 105
Cars loaded so far: 126
Cars loaded so far: 147
Cars loaded so far: 168
Cars loaded so far: 189
Cars loaded so far: 210
Cars loaded so far: 231
Cars loaded so far: 252
Cars loaded so far: 273
Cars loaded so far: 294
Reached target of 293 cars. Stopping.
Total cars to extract: 293

 Extraction done!

saved as cardhekho_new_cars.csv


### Data Collection

Used Selenium to scrape CarDekho's "New Cars" listing page, which loads results 
incrementally via a "Load More" button rather than standard pagination. The scraper 
detects the total listed car count (293) directly from the page and clicks "Load More" 
repeatedly until that target is reached, with a streak-based fallback to stop early if 
the page stops returning new cars. For each car card, five raw fields are extracted: 
name, rating, price range, mileage, and engine displacement — saved to **cardhekho_new_cars.csv**.